<a href="https://colab.research.google.com/github/Nerneri/World_bank_PIP_analysis/blob/lisa_branch/Poverty_And_Inequality_Platform_(PIP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Poverty And Inequality Platform Analysis Project**



First dataset page. Poverty And Inequality Platform (PIP) : https://datacatalog.worldbank.org/search/dataset/0063646/poverty-and-inequality-platform-pip-percentiles$0

Second dataset page. World Bank Country and Lending Groups : https://www.kaggle.com/datasets/taniaj/world-bank-country-and-lending-groups?resource=download$0







In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data_path_PIP = '/content/drive/MyDrive/python_project/world_100bin_revised_filtered_2015.csv'
data_path_classification = '/content/drive/MyDrive/python_project/worldbank_classification.csv'

data_PIP = pd.read_csv(data_path_PIP)
data_classification = pd.read_csv(data_path_classification)

# **1. Abstract/Annotation**

This notebook Is dedicated to analysis of “The Poverty and Inequality Platform: Percentiles” dataset. The analysis is based on two datasets, the PIP Percentiles dataset from the World Bank containing welfare distribution data for 152 countries over 1963-2025, and the World Bank Country and Lending Groups dataset providing regional and income group classifications for 218 economies.
We have made 4 hypothesises:
* Inequality ratio has decreased over the last decade.
* The median avg_welfare differs significantly between consumption-based and income-based surveys (welfare_type).
* Sub-Saharam Africa has the lowest average welfare compared to other regions.
* The gap between top 10% and bottom 10% welfare has grown over time.

Roles were distributed as follows:
* Alexandra Khaidukova: Dataset description, data cleanup of PIP dataset, descriptive statistics, data transformation, hypothesises check.
* Alisa Narozhnaya: Data cleanup of World Bank Country and Lending Groups dataset, creation of plots, their overview and comparison, hypothesises check.




---



# **2. Dataset description**

### Poverty And Inequality Platform dataset desciption

The Poverty and Inequality Platform: Percentiles (hereafter referred as PIP) reports 100 points ranked according to the consumption or income distribution for country-year survey data available in the World Bank’s Poverty and Inequality Platform. This analysis is based on a shortened version of original dataset with year range 2015-2025 instead of 1963-2025.

Each distribution reports 100 points per country per survey year ranked from the smallest (percentile 1) to the largest (percentile 100) income or consumption. For each income percentile, the database reports the following variables: the average daily per person income or consumption (avg_welfare); the income or consumption value for the upper threshold of the percentile (quantile); the share and total population of the percentile (pop_share, pop); and the share of income or consumption held by each percentile (welfare_share). In addition, the database reports the welfare measure (welfare_type) used in the survey data—income or consumption—and the region covered (reporting_level)—urban, rural, or national. The distributions are available in 2017 or 2021 PPP$ (Purchasing Power Parity).



In [ ]:
print(f'Rows:    {data_PIP.shape[0]:,}')
print(f'Columns: {data_PIP.shape[1]}')
display(data_PIP.head(10))

In [ ]:
data_PIP.info()

There are missing values in quantile.

Moreover, dtype of country_code, reporting_level and welfare_type is object (types of values are mixed).

We will correct this further in 3. Data cleanup.


In [ ]:
print(f"Countries: {data_PIP['country_code'].nunique()}")
print(f"Year range: {data_PIP['year'].min()} – {data_PIP['year'].max()}")
print(f"Reporting levels: {data_PIP['reporting_level'].unique()}")
print(f"Welfare types: {data_PIP['welfare_type'].unique()}")
print(f"Percentiles: {data_PIP['percentile'].min()} – {data_PIP['percentile'].max()}")

### World Bank Country and Lending Groups dataset desciption


Dataset World Bank Country and Lending Groups have 7 columns - a row index (x), country (Economy), country code (Code), region of the country (Region), Income group and Lending category according to World Bank classification for 2026 and Other. The dataset covers 7 unique regions, 4 income group tiers (low income, lower middle income, upper middle income, high income) and 2 lending types (consumption and income) based on 2026 GNI per capita thresholds. It is planned to be merged with the PIP dataset on country code, adding regional and income context to 152 matched economies.

In [ ]:
print(f'Rows:    {data_classification.shape[0]:,}')
print(f'Columns: {data_classification.shape[1]}')
data_classification.head(10)

We can notice that Income group and Lending category require renaming.
There is no need in having column x, as we already have basic pd indexes. Moreover, we will delete 'Lending category' and 'Other', since they are not that important for our analysis, besides having a lot of missing values.
It also worth mentioning, that in this dataset income group classification reflects 2026 fiscal year definitions, based on 2024 GNI data.

In [ ]:
data_classification.info()

Only x(id) have an explicit dtype, we will also correct this later.



In [ ]:
print(f"Countries: {data_classification['Economy'].nunique()}")
print(f"Income levels: {data_classification['Income group'].unique()}")



---



# **3. Data cleanup**

In [ ]:
import missingno

missingno.matrix(data_PIP)

In [ ]:
data_PIP = data_PIP.dropna(subset=['quantile']).reset_index(drop=True)
data_PIP.info()

As country_code, reporting_level and welfare_type does not have missing values, we can just forse their dtype into string:

In [ ]:
for col in ['country_code', 'reporting_level', 'welfare_type']:
    data_PIP[col] = data_PIP[col].astype('string')

In [ ]:
data_PIP.info()


In [ ]:
missingno.matrix(data_classification)

Rename columns:

In [ ]:
data_classification = data_classification.rename(columns=lambda name: name.lower().replace(' ', '_'))

In [ ]:
data_classification.rename(columns={'economy': 'country'}, inplace=True)

In [ ]:
for col in ['country', 'code', 'region', 'income_group']:
    data_classification[col] = data_classification[col].astype('string')

In [ ]:
data_classification = data_classification.drop(['lending_category', 'other', 'x'], axis = 1).reset_index(drop=True)

In [ ]:
data_classification.info()



---



# *. Creation of new dataset

Now we can create new dataset with regions instead of countries. Intersection of two datasets on the number of countries is right - all 152 countries from PIP dataset are in classification dataset.
Then we will remove dublicate of code and sort it by region:

In [ ]:
pip_codes = set(data_PIP['country_code'].unique())
class_codes = set(data_classification['code'].unique())
print(len(pip_codes))
print(len(class_codes))
print(len(pip_codes & class_codes))

In [ ]:
data_PIP_wregion = data_PIP.merge(data_classification, left_on = 'country_code',
                                 right_on = 'code', how='left')
data_PIP_wregion = data_PIP_wregion.drop(['code'], axis = 1)
data_PIP_wregion = data_PIP_wregion.sort_values('year').reset_index(drop =True)

In [ ]:
data_PIP_wregion.head()



---



# **5. Descriptive statistics**

In [ ]:
NUMERIC = ['avg_welfare', 'pop_share', 'welfare_share', 'pop']
data_PIP_wregion[NUMERIC].describe().round(2)

Below we can see balance across income groups - how many entries belong to each group:

In [ ]:
data_PIP_wregion['income_group'].value_counts()

Same as above, but in percentage:

In [ ]:
data_PIP_wregion['income_group'].value_counts(normalize=True)

Standart deviation of pop_share and welfare_share are minimal (0.00 and 0.01 respectevely), which implies that population and welfare shares are very evenly distributed across percentiles. This was expected, as by dataset design each row represents exactly one percentile.

In contrast, for averge welfare deviation is 32.20, whereas for population is 1843836.69. So in this situation, there is large difference in welfare levels between countries and percentiles. Hight std for pop is expected, since population varies enourmasly (for example population of India or China is much bigger than population of small islands, for example Marshall islands).

In [ ]:
data_PIP_wregion[NUMERIC].std().round(2)

Variance confirm same pattern, as in std, but in squared units:

In [ ]:
data_PIP_wregion[NUMERIC].var().round(2)

North America has the highest mean welfare (78.79) followed by Europe & Central Asia (38.70) and Middle East & North Africa (30.50). South Asia (9.81) and Sub-Saharan Africa (5.36) have the lowest mean welfare highlighting the extreme global welfare gap.
The difference between mean and median avg_welfare within each region indicates skewness. In other words, some wealthy countries pull the regional mean above the median. Larger the difference is - the larger the internal inequality within the region.

Standard deviation of avg_welfare is highest in North America (55.04) and Middle East & North Africa (31.94), meaning these regions have the most internally diverse welfare levels. Sub-Saharan Africa despite having the lowest mean also has relatively low std (5.94), suggesting countries in the region are more overall poor rather than having large internal differences.

In [ ]:
data_PIP_wregion.groupby('region')[NUMERIC].agg(['mean', 'median', 'std']).round(2)



---



# **6. Data transofrmation**

Create new columns = inequality_ratio, decade and welfare_delta

In [ ]:
data_PIP_wregion['inequality_ratio'] = data_PIP_wregion['welfare_share'] / data_PIP_wregion['pop_share']

In [ ]:
data_PIP_wregion['decade'] = (data_PIP_wregion['year'] // 10) * 10


In [ ]:
data_PIP_wregion = data_PIP_wregion.sort_values(['country_code', 'year', 'percentile'])
data_PIP_wregion['welfare_delta'] = data_PIP_wregion.groupby(
    ['country_code', 'year']
)['avg_welfare'].diff().fillna(0)

In [ ]:
data_PIP_wregion.head()

Create new dataset and bin abg_welfare into 3 categories based on daily welfare per person value:

In [ ]:
categorical_welfare = data_PIP_wregion.copy().reset_index(drop=True)

In [ ]:

bins = pd.IntervalIndex.from_tuples(
    [(0, 1), (1, 5), (5, np.inf)], closed='left'
)
categorical_welfare['welfare_category'] = pd.cut(categorical_welfare['avg_welfare'], bins)
categorical_welfare['welfare_category'] = categorical_welfare['welfare_category'].cat.rename_categories(
    ['low', 'medium', 'high']
)

In [ ]:
print(categorical_welfare['welfare_category'].unique())

In [ ]:
mapper = {
    'low' : 0,
    'medium' : 1,
    'high' : 2,
}

In [ ]:
categorical_welfare['welfare_category_ord'] = categorical_welfare['welfare_category'].map(mapper)

This shows inequality ratio (welfare / pop_share), where ratio < 1 means percentile recieves less than its population share, ratio > 1 percentile recieves more than its population share, ratio = 0 indicates perfectly equal distribution of welfare and pop_share.

In [ ]:
categorical_welfare.head()



---



# **7. Plots**

1. Histogram. Distribution of Average Welfare.

The distribution of average welfare is right-skewed, most observations fall below 50.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(data_PIP_wregion['avg_welfare'], bins=50, edgecolor='black', alpha=0.6)
ax.set_title('Distribution of Average Welfare')
ax.set_xlabel('avg_welfare')
ax.set_ylabel('count')
plt.show()

2. Line plot. Average Welfare Over Time.

Average welfare rose from 2014 to 2020, reaching its peak (~35). After 2020 it fluctuated, then dropped sharply to ~19-20 in 2024-2025 (due to fewer data reported for these years).

In [ ]:
data_PIP_wregion.groupby('year').size()

In [ ]:
data_PIP_wregion.groupby('year')['avg_welfare'].mean().plot(figsize=(10,5), title='Average Welfare Over Time')
plt.ylabel('avg_welfare')
plt.show()

3. Scatter Plot. Average Welfare vs Inequality Ratio.

At low average welfare levels, the inequality ratio varies from near 0 to over 7.  As average welfare increases, the spread narrows, with high-welfare points clustering around 3-4.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(data_PIP_wregion['avg_welfare'], data_PIP_wregion['inequality_ratio'], alpha=0.1)
ax.set_title('Average Welfare vs Inequality Ratio')
ax.set_xlabel('avg_welfare')
ax.set_ylabel('inequality_ratio')
plt.show()

4. Boxplot. Average Welfare by Income Group


Median average welfare increases with income group: low income countries are clustered near 0, while high income countries show a much higher median (~50) and a wider spread, with outliers reaching close to 400.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=data_PIP_wregion, x='income_group', y='avg_welfare', ax=ax,
            order=['Low income', 'Lower middle income', 'Upper middle income', 'High income'])
ax.set_title('Average Welfare by Income Group')
plt.xticks(rotation=15)
plt.show()



---




# **8. Detailed overview**

Comparing Plot 1. Average Welfare by Region Over Time.

North America is the highest (~70-90) and rising. Europe & Central Asia is stable around 35-45 until a sharp drop to near 0 around 2024 (due to less data reported again). Middle East & North Africa spikes to ~48 then drops. East Asia & Pacific declines from ~25 to ~10. South Asia and Sub-Saharan Africa stay low (0-15) throughout. South Asia has highly irregular year coverage (data is completely absent for 2020, 2021, 2023 and 2025) which is why South Asia line appears only for part of the timeline.

In [ ]:
data_PIP_wregion.groupby(['year', 'region'])['avg_welfare'].mean().unstack().plot(figsize=(12, 6))
plt.title('Average Welfare by Region Over Time')
plt.show()

In [ ]:
data_PIP_wregion[data_PIP_wregion['region'] == 'Europe & Central Asia'].groupby('year').size()

In [ ]:
data_PIP_wregion[data_PIP_wregion['region'] == 'South Asia'].groupby('year').size()

Comparing plot 2. Top 10% vs bottom 10% Welfare Over Time.

Top 10% line fluctuates between ~63-80, peaking around 2020 and 2023, then drops sharply to ~57 by 2024-2025. Bottom 10% stays nearly flat at ~8-10 throughout, with a slight dip at the end too.

In [ ]:
top10 = data_PIP_wregion[data_PIP_wregion['percentile'] >= 90].groupby('year')['avg_welfare'].mean()
bot10 = data_PIP_wregion[data_PIP_wregion['percentile'] <= 10].groupby('year')['avg_welfare'].mean()
plt.figure(figsize=(12,5))
plt.plot(top10, label='Top 10%')
plt.plot(bot10, label='Bottom 10%')
plt.legend()
plt.title('Top 10% vs bottom 10% Welfare Over Time')
plt.show()

Comparing plot 3. Inequality Ratio by Income Group and Decade

All four income groups (Lower middle, Upper middle, High, Low) have very similar median inequality ratios (~0.7-1.0) in both the 2010s and 2020s, with lots of high outliers in every group.

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
sns.boxplot(data=data_PIP_wregion, x='decade', y='inequality_ratio', hue='income_group', ax=ax)
ax.set_title('Inequality Ratio by Income Group and Decade')
plt.show()

Comparing plot 4. Average Welfare Shareby by region vs Average Welfare Shareby by income group

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
data_PIP_wregion.groupby('region')['avg_welfare'].mean().plot(kind='bar', ax=ax1, title='Avg Welfare by Region')
data_PIP_wregion.groupby('income_group')['avg_welfare'].mean().plot(kind='bar', ax=ax2, title='Avg Welfare by Income Group')
plt.tight_layout()
plt.show()

Comparing plot 5. Welfare Share vs Pop Share by Region

Both bar charts look nearly identical — all regions have welfare_share and pop_share both around 0.008-0.010, with very little visible difference between regions or between the two charts.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

data_PIP_wregion.groupby('region')['welfare_share'].mean().plot(
    kind='bar', ax=ax1, title='Welfare Share by Region', color='steelblue'
)
data_PIP_wregion.groupby('region')['pop_share'].mean().plot(
    kind='bar', ax=ax2, title='Pop Share by Region', color='steelblue'
)

ax1.set_xlabel('region')
ax1.set_ylabel('welfare_share mean')
ax2.set_xlabel('region')
ax2.set_ylabel('pop_share mean')

plt.tight_layout()
plt.show()



---



# **9. Hypothesis check**

### **Hypothesis 1: Inequality ratio decreased over the past decade.**

This hypothesis is rejected. Although there is a slight decrease berween 2015 (0.945561) and 2025 (0.944464), the change is approximately 0.001, which is neglible and cannot be considered significant.

The inequality remained stable (around 0.9485) throughout the entire period.

In [ ]:
data_PIP_wregion.groupby('year')['inequality_ratio'].mean().plot(kind='bar', figsize=(8,5))
plt.title('Mean Inequality Ratio by Year')
plt.show()
print(data_PIP_wregion.groupby('year')['inequality_ratio'].mean())


### **Hypothesis 2: High-income countries have lower inequality ratio**

This hypothesis is false. Hight income countries actually have the highest median inequality ratio (0.844419), compared to low income countries (0.749660).

All four income groups show very similar distributions with medians close to 1, suggesting that inequality ratio does not differ significantly across income groups.
The large number of outliers in all groups indicates that within each income group there are countries with extreme inequality ratios regardless of their income level. Hypothesis 2 is rejected — high-income countries do not have a lower inequality ratio than low-income countries.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=data_PIP_wregion,
    x='income_group',
    y='inequality_ratio',
    order=['Low income', 'Lower middle income', 'Upper middle income', 'High income'],
    ax=ax
)
ax.set_title('Inequality Ratio by Income Group')
ax.set_xlabel('Income Group')
ax.set_ylabel('Inequality Ratio')
plt.xticks(rotation=15)
plt.show()

# print stats to confirm
print(data_PIP_wregion.groupby('income_group')['inequality_ratio'].median().sort_values())


### **Hypothesis 3: Sub-Saharam Africa has the lowest average welfare compared to other regions.**

This hypothesis is true, Sub-Saharam Africa region has the lowest average welfare, standing at 5.359984.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
data_PIP_wregion.groupby('region')['avg_welfare'].mean().plot(
    kind='bar', ax=ax, color='steelblue'
)
ax.set_title('Average Welfare by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Mean avg_welfare')
plt.xticks(rotation=15)
plt.show()

print(data_PIP_wregion.groupby('region')['avg_welfare'].mean().sort_values())

### **Hypothesis 4: The gap between top 10% and bottom 10% welfare has grown over time.**

Gap in 2025 < gap in 2025. Thus the hypothesis is false.

In [ ]:
top10 = data_PIP_wregion[data_PIP_wregion['percentile'] >= 90].groupby('year')['avg_welfare'].mean()
bot10 = data_PIP_wregion[data_PIP_wregion['percentile'] <= 10].groupby('year')['avg_welfare'].mean()
plt.figure(figsize=(12,5))
plt.plot(top10, label='Top 10%')
plt.plot(bot10, label='Bottom 10%')
plt.legend()
plt.title('Top 10% vs bottom 10% Welfare Over Time')
plt.show()

print(f"  bottom in 2015: {bot10[2015]:.2f}")
print(f"  top in 2015: {top10[2015]:.2f}")
print(f"  gap in 2015: {top10[2015] - bot10[2015]:.2f}")

print(f"  bottom in 2025: {bot10[2025]:.2f}")
print(f"  top in 2025: {top10[2025]:.2f}")
print(f"  gap in 2025: {top10[2025] - bot10[2025]:.2f}")